# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

- 관계 [ 수긍 : 추락 = 대사관 : ? ]
- 모델이 예측한 가장 적합한 단어: 잠입
- 당신의 답변과 모델 예측의 유사도: 0.34
- 아쉽네요. 더 생각해보세요.

In [99]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [100]:
df = df['sentence']

In [101]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [111]:
from konlpy.tag import Okt
from tqdm import tqdm

okt = Okt()
ko_stopwords = ["은", "는", "이", "가", "을", "를", "과", "와", "들", "도", "부터", "까지", "에", "나", "너", "그", "걔", "얘"]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)
    sentence = re.sub(r"[^가-힣\s]", " ", sentence)
    tokens = okt.morphs(sentence, stem=True)
    tokens = [token for token in tokens if token not in ko_stopwords]
    preprocessed_data.append(tokens)

100%|██████████| 10000/10000 [00:15<00:00, 647.30it/s]


In [103]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data, # corpus
    vector_size=100,                  # 임베딩 벡터 차원
    sg=0,                             # 학습 알고리즘 (0:CBOW, 1:Skip-gram)
    window=5,                         # 주변 단어 수 (앞뒤로 n개 사용) -> 이게 왜 5로 설정했는지 다시 확인
    min_count=5                       # 최소 빈도
)

model.wv.vectors.shape

(4060, 100)

In [104]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
하다,-0.287034,0.371382,0.086437,0.281889,-0.121598,-0.384214,0.356629,0.745411,-0.339086,-0.494405,...,-0.089013,0.248728,0.155195,-0.073003,0.939983,0.290067,0.539805,-0.382342,0.179902,0.120315
의,-0.289110,0.625725,0.071752,0.202563,0.093021,-0.389603,0.693657,0.625319,-0.265185,-0.271078,...,0.138163,0.190407,0.141507,-0.068927,0.781018,0.134878,0.716191,-0.156812,0.024894,0.048788
있다,-0.405497,0.209871,0.099042,0.218269,-0.110611,-0.536660,0.087258,0.916483,-0.435741,-0.704605,...,-0.115629,0.418330,0.229043,0.002311,1.063069,0.390536,0.542218,-0.718257,0.253110,0.068256
에서,-0.227268,0.664818,0.026193,0.249846,0.138119,-0.265234,0.798356,0.564125,-0.265375,0.076064,...,0.193476,0.252033,0.129938,-0.198798,0.682576,0.128835,0.693865,0.051275,0.139186,-0.036835
으로,-0.368621,0.233913,0.003053,0.145196,-0.214832,-0.489652,0.248182,0.878988,-0.443405,-0.829967,...,-0.102677,0.223810,0.231609,-0.002714,1.022089,0.246262,0.662028,-0.607957,0.187303,0.070128
이다,-0.402931,0.372206,0.072618,0.216539,-0.012191,-0.613085,0.402867,0.950892,-0.418476,-0.691732,...,0.108400,0.240966,0.178407,0.020632,1.021489,0.293085,0.558913,-0.542021,0.107332,0.100407
한,-0.229693,0.482623,0.044145,0.197790,-0.051367,-0.345663,0.549620,0.641368,-0.297339,-0.416420,...,-0.001052,0.206489,0.141135,-0.043866,0.845897,0.155531,0.653028,-0.271745,0.075418,0.082688
로,-0.291950,0.494775,0.105300,0.301601,0.028720,-0.343694,0.546483,0.687459,-0.315617,-0.229565,...,0.093017,0.282155,0.134653,-0.049676,0.752710,0.233775,0.600218,-0.221566,0.161728,0.079518
일,-0.036678,1.245992,-0.116933,0.485599,0.336502,0.140459,1.662797,0.168729,-0.296359,1.021199,...,0.393948,0.207010,0.144459,-0.592283,0.428991,-0.026223,0.961829,1.011817,0.307441,-0.155407
되다,-0.276670,0.468695,0.045438,0.261631,-0.072214,-0.383919,0.496729,0.709978,-0.344311,-0.363231,...,-0.006106,0.279271,0.159511,-0.077370,0.860784,0.186117,0.640212,-0.223148,0.189340,0.064536


In [105]:
# 학습된 단어 임베딩 저장
model.wv.save_word2vec_format('all_kor_w2v')

In [106]:
# 임베딩 모델 로드
from gensim.models import KeyedVectors

load_model = KeyedVectors.load_word2vec_format('all_kor_w2v')

In [107]:
# model : Word2Vec
model.wv.most_similar('남자')

[('맞다', 0.9994053840637207),
 ('알다', 0.9993279576301575),
 ('식', 0.9993200302124023),
 ('법', 0.9993120431900024),
 ('일부', 0.9993049502372742),
 ('함', 0.999297559261322),
 ('커피', 0.9992868900299072),
 ('기', 0.9992827773094177),
 ('먹다', 0.9992708563804626),
 ('대신', 0.9992696046829224)]

In [108]:
import random

kv = load_model  # 이미 불러온 임베딩 사용

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B, C 단어 랜덤 선택
    A, B, C = random.sample(vocab, 3)

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("해당 단어들로는 예측 불가. 다시 실행하세요.")
        return

    # 출력
    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    # 사용자 입력
    user = input("D를 입력하세요").strip()
    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")


In [122]:
play()

관계 [ 주고받다 : 랑 = 보고 : ? ]
모델이 예측한 가장 적합한 단어: 때문
당신의 답변과 모델 예측의 유사도: 0.99
